# 생성형 인공지능 1주차 실습 — Diffusion Model (DDPM / Score-based / SDE)

**[문제 노트북]**

---

이 노트북은 1주차 강의자료 *「생성형 인공지능 — Diffusion」* 의 내용을 직접 코드로 구현하는 실습입니다.
강의에서 다룬 아래 수식들을 하나씩 PyTorch로 옮깁니다.

| 강의 수식 | 실습 문제 |
|---|---|
| $q(\mathbf{x}_t\vert\mathbf{x}_{t-1})=\mathcal N(\mathbf{x}_t;\sqrt{1-\beta_t}\mathbf{x}_{t-1},\beta_t\mathbf I)$ | 문제 1 |
| $q(\mathbf{x}_t\vert\mathbf{x}_0)=\mathcal N(\mathbf{x}_t;\sqrt{\bar\alpha_t}\mathbf{x}_0,(1-\bar\alpha_t)\mathbf I)$ | 문제 2 |
| $q(\mathbf{x}_{t-1}\vert\mathbf{x}_t,\mathbf{x}_0)=\mathcal N(\mathbf{x}_{t-1};\tilde\mu_t,\tilde\beta_t\mathbf I)$ | 문제 3 |
| $\mathbb E_{t,\mathbf{x}_0,\epsilon}\big[\lambda(t)\lVert\epsilon-\epsilon_\theta(\mathbf{x}_t,t)\rVert^2\big]$ | 문제 4 |
| $p_\theta(\mathbf{x}_{t-1}\vert\mathbf{x}_t)$ 역과정 샘플링 | 문제 5 |
| $\mathbf{s}_\theta(\mathbf x,t)\approx\nabla_{\mathbf x}\log q(\mathbf x)$, Langevin 동역학 | 문제 6 |
| DDIM (가속 샘플링) | 문제 7 (보너스) |

**실행 환경**: Colab → 런타임 → 런타임 유형 변경 → **T4 GPU** 를 선택하세요.
(GPU가 없으면 아래 설정 셀에서 `QUICK_MODE = True` 로 두면 CPU에서도 동작합니다. 대신 생성 품질은 낮습니다.)


> ### 제출 안내
> - `# TODO:` 주석이 있는 곳과 `raise NotImplementedError(...)` 줄을 지우고 직접 구현하세요.
> - 각 문제 뒤의 **자동 채점 셀**을 실행해 `문제 N 통과` 가 출력되면 정답입니다.
> - 채점 셀은 수정하지 마세요.
> - 서술형 문항은 셀 아래에 마크다운 셀을 추가해 답을 적으세요.
> - 총 7문제 (문제 7은 보너스), 배점은 문제당 동일합니다.


## 0. 환경 설정 (그대로 실행)

In [ ]:
import math, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

# ------------------------------------------------------------------
# 실습 전역 설정
# ------------------------------------------------------------------
T = 400                 # 확산 스텝 수 (마르코프 체인의 길이)
IMG_SIZE = 32           # MNIST 를 32x32 로 패딩해서 사용 (다운샘플링이 깔끔해짐)
CHANNELS = 1

QUICK_MODE = False      # True 로 두면 아주 짧게 학습 (CPU 확인용)
BATCH_SIZE = 128
EPOCHS      = 2 if QUICK_MODE else 12
N_TRAIN     = 4000 if QUICK_MODE else 60000
BASE_CH     = 32 if QUICK_MODE else 64

In [ ]:
# 텐서를 배치 차원에 맞춰 뽑아주는 헬퍼 (문제에서 반복 사용)
def extract(a, t, x_shape):
    """1차원 스케줄 텐서 a 에서 시점 t 의 값을 뽑아 x_shape 에 브로드캐스트 가능하게 reshape.

    a       : (T,)      스케줄 텐서
    t       : (B,)      long 텐서, 각 샘플의 시점
    x_shape : 브로드캐스트 대상 텐서의 shape  (예: (B, 1, 32, 32))
    return  : (B, 1, 1, 1)
    """
    b = t.shape[0]
    out = a.to(t.device).gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))


def show_images(imgs, title="", nrow=8):
    """(B,1,H,W), 값 범위 [-1,1] 인 텐서를 그리드로 표시."""
    imgs = imgs.detach().cpu().clamp(-1, 1)
    imgs = (imgs + 1) / 2
    n = imgs.shape[0]
    ncol = nrow
    nrows = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrows, ncol, figsize=(ncol * 1.1, nrows * 1.1))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

print("헬퍼 준비 완료")

---
## 문제 1. 노이즈 스케줄 $\beta_t,\ \alpha_t,\ \bar\alpha_t$ (난이도 ★☆☆)

강의자료의 forward process 는

$$q(\mathbf{x}_t\mid\mathbf{x}_{t-1})=\mathcal N\!\left(\mathbf{x}_t;\sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\ \beta_t\mathbf I\right)$$

로 정의되고, 여기서 $\beta_t$ 는 **사람이 미리 정하는(학습하지 않는) 하이퍼파라미터 스케줄** 입니다.
강의 코드에는 두 가지 스케줄이 등장했습니다.

**(1) linear schedule** — 노이즈 강도가 선형으로 증가
$$\beta_t=\text{linspace}(\beta_{\text{start}},\ \beta_{\text{end}},\ T),\qquad
\beta_{\text{start}}=\tfrac{1000}{T}\times10^{-4},\quad \beta_{\text{end}}=\tfrac{1000}{T}\times 0.02$$

**(2) cosine schedule** (Improved DDPM) — $\bar\alpha_t$ 자체를 코사인 곡선으로 설계
$$\bar\alpha_t=\frac{f(t)}{f(0)},\qquad f(t)=\cos^2\!\left(\frac{t/T+s}{1+s}\cdot\frac{\pi}{2}\right),\qquad
\beta_t=1-\frac{\bar\alpha_t}{\bar\alpha_{t-1}}$$

### 할 일
1. `linear_beta_schedule`, `cosine_beta_schedule` 을 완성하세요.
2. $\alpha_t=1-\beta_t$, $\bar\alpha_t=\prod_{s=1}^{t}\alpha_s$ 를 계산하세요.
3. 두 스케줄의 $\bar\alpha_t$ 를 그려 비교하고, **cosine 스케줄이 중간 구간에서 정보를 더 오래 보존하는 이유**를 설명하세요.

> 힌트: `torch.linspace`, `torch.cumprod`, `torch.cos`, `torch.clip`


In [ ]:
def linear_beta_schedule(timesteps):
    """선형 베타 스케줄. return: (T,) float64 텐서"""
    # TODO: scale = 1000 / timesteps 로 두고 beta_start, beta_end 를 정의한 뒤
    # TODO: torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float64) 를 반환
    raise NotImplementedError("여기를 구현하세요")


def cosine_beta_schedule(timesteps, s=0.008):
    """코사인 베타 스케줄 (Improved DDPM). return: (T,) float64 텐서"""
    # TODO: steps = timesteps + 1 개의 격자 x 를 만들고
    # TODO: alphas_cumprod = cos(((x/timesteps) + s) / (1+s) * pi * 0.5) ** 2 를 계산
    # TODO: alphas_cumprod 를 alphas_cumprod[0] 으로 나눠 정규화 (abar_0 = 1)
    # TODO: betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1] 로 되돌리고 [0, 0.999] 로 clip
    raise NotImplementedError("여기를 구현하세요")


# ---- 실습 전반에서 사용할 스케줄 (linear) -------------------------------
# [중요] t 가 작을 때 (1 - abar_t) 는 자리수 손실(catastrophic cancellation)이 크다.
#        따라서 스케줄 계산은 float64 로 하고, 마지막에만 float32 로 캐스팅한다.
betas64 = linear_beta_schedule(T)          # float64, CPU

# TODO: alphas64 = 1 - betas64
# TODO: alphas_cumprod64 = alphas64 의 누적곱 (torch.cumprod)
# TODO: alphas_cumprod_prev64 = abar 를 오른쪽으로 한 칸 민 것 (맨 앞은 1.0). F.pad(..., (1,0), value=1.0)
raise NotImplementedError("여기를 구현하세요")

def to32(x):
    return x.float().to(device)

betas               = to32(betas64)
alphas              = to32(alphas64)
alphas_cumprod      = to32(alphas_cumprod64)
alphas_cumprod_prev = to32(alphas_cumprod_prev64)

# 앞으로 자주 쓰는 파생 상수들 (미리 계산해 두면 샘플링이 빨라짐)
sqrt_alphas_cumprod           = to32(torch.sqrt(alphas_cumprod64))
sqrt_one_minus_alphas_cumprod = to32(torch.sqrt(1.0 - alphas_cumprod64))
sqrt_recip_alphas_cumprod     = to32(torch.sqrt(1.0 / alphas_cumprod64))
sqrt_recipm1_alphas_cumprod   = to32(torch.sqrt(1.0 / alphas_cumprod64 - 1))

print("beta[0] = %.6f,  beta[-1] = %.4f" % (betas[0], betas[-1]))
print("abar[0] = %.6f,  abar[-1] = %.6f" % (alphas_cumprod[0], alphas_cumprod[-1]))

In [ ]:
# ===================== 문제 1 자동 채점 =====================
_bl = linear_beta_schedule(T)
_bc = cosine_beta_schedule(T)
assert _bl.shape == (T,) and _bc.shape == (T,), "스케줄 길이가 T 여야 합니다"
assert torch.all(_bl > 0) and torch.all(_bl < 1), "beta 는 (0,1) 범위여야 합니다"
assert torch.all(_bc >= 0) and torch.all(_bc <= 0.999), "cosine beta clip 범위 확인"
assert torch.all(_bl[1:] >= _bl[:-1]), "linear beta 는 단조 증가해야 합니다"

_ac = torch.cumprod(1 - _bc, dim=0)
assert torch.all(alphas_cumprod[1:] <= alphas_cumprod[:-1] + 1e-8), "abar 는 단조 감소해야 합니다"
assert abs(alphas_cumprod[0].item() - (1 - betas[0].item())) < 1e-6, "abar_0 = alpha_0 여야 합니다"
assert alphas_cumprod[-1].item() < 1e-3, "abar_T 는 0 에 가까워야 합니다 (완전한 노이즈)"
assert abs(alphas_cumprod_prev[0].item() - 1.0) < 1e-6, "abar_{-1} 은 1 이어야 합니다"
assert torch.allclose(alphas_cumprod_prev[1:], alphas_cumprod[:-1]), "abar_prev 는 abar 를 한 칸 민 값"
# cosine 스케줄이 중간 구간에서 정보를 더 오래 보존하는지
assert _ac[T // 2] > torch.cumprod(1 - _bl, dim=0)[T // 2], "중간 구간 abar 는 cosine 이 더 커야 합니다"
print("문제 1 통과")

In [ ]:
# 시각화 — 두 스케줄의 abar 비교
ab_lin = torch.cumprod(1 - linear_beta_schedule(T), dim=0)
ab_cos = torch.cumprod(1 - cosine_beta_schedule(T), dim=0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(linear_beta_schedule(T), label="linear")
ax[0].plot(cosine_beta_schedule(T), label="cosine")
ax[0].set_title(r"$\beta_t$"); ax[0].set_xlabel("t"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ab_lin, label="linear")
ax[1].plot(ab_cos, label="cosine")
ax[1].set_title(r"$\bar{\alpha}_t$  (signal retained)"); ax[1].set_xlabel("t")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**해설 (문제 1-3).**
$\bar\alpha_t$ 는 $\mathbf x_t=\sqrt{\bar\alpha_t}\mathbf x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$ 에서
**원본 신호가 남아 있는 비율**을 뜻합니다.
linear 스케줄은 $\bar\alpha_t$ 가 초반에 급격히 0 으로 떨어져서, 체인의 뒷부분 대부분이 "이미 거의 순수 노이즈"인
정보량 없는 구간이 됩니다. 반면 cosine 스케줄은 $\bar\alpha_t$ 를 코사인 제곱 곡선으로 **직접 설계**하여
중간 구간에서 완만하게 감소시키므로, 학습에 유용한 신호대잡음비(SNR) 구간에 스텝이 더 고르게 배분됩니다.
그 결과 특히 저해상도 이미지에서 샘플 품질과 로그우도가 개선됩니다.


---
## 문제 2. Forward Diffusion 의 닫힌 형식 $q(\mathbf x_t\mid\mathbf x_0)$ (난이도 ★☆☆)

$T$ 번 노이즈를 순차적으로 더하는 대신, 가우시안의 재매개변수화를 이용하면 **한 번에** 임의 시점 $t$ 로 점프할 수 있습니다.

$$q(\mathbf x_t\mid\mathbf x_0)=\mathcal N\!\left(\mathbf x_t;\sqrt{\bar\alpha_t}\mathbf x_0,\ (1-\bar\alpha_t)\mathbf I\right)
\qquad\Longleftrightarrow\qquad
\boxed{\ \mathbf x_t=\sqrt{\bar\alpha_t}\,\mathbf x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathcal N(0,\mathbf I)\ }$$

이 성질 덕분에 DDPM 학습에서 매 스텝마다 $t$ 를 균등하게 뽑아 **한 번의 forward 로** 손실을 계산할 수 있습니다.

### 할 일
1. `q_sample(x0, t, noise)` 을 구현하세요. (`extract` 헬퍼를 사용)
2. MNIST 이미지 한 장이 $t$ 가 커질수록 어떻게 망가지는지 시각화하세요.


In [ ]:
def q_sample(x0, t, noise=None):
    """x0 (B,C,H,W) 를 시점 t (B,) 로 한 번에 확산시킨다. return: x_t (B,C,H,W)"""
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO: extract(sqrt_alphas_cumprod, t, x0.shape) * x0
    # TODO:   + extract(sqrt_one_minus_alphas_cumprod, t, x0.shape) * noise
    raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ===================== 문제 2 자동 채점 =====================
_B = 100000
_x0 = torch.full((_B, 1, 1, 1), 0.7, device=device)      # 상수 이미지
for _tv in [0, T // 4, T // 2, T - 1]:
    _t = torch.full((_B,), _tv, device=device, dtype=torch.long)
    _xt = q_sample(_x0, _t)
    _m_emp, _s_emp = _xt.mean().item(), _xt.std().item()
    _m_true = (sqrt_alphas_cumprod[_tv] * 0.7).item()
    _s_true = sqrt_one_minus_alphas_cumprod[_tv].item()
    assert abs(_m_emp - _m_true) < 0.03, f"t={_tv}: 평균이 sqrt(abar_t)*x0 와 다릅니다"
    assert abs(_s_emp - _s_true) < 0.03, f"t={_tv}: 표준편차가 sqrt(1-abar_t) 와 다릅니다"

# noise 를 직접 넣었을 때 결정론적으로 같은 값이 나오는지
_x0 = torch.randn(4, 1, 8, 8, device=device)
_n  = torch.randn_like(_x0)
_t  = torch.tensor([0, 10, 100, T - 1], device=device)
assert torch.allclose(q_sample(_x0, _t, _n), q_sample(_x0, _t, _n)), "noise 인자가 무시되고 있습니다"
# t=0 이면 거의 원본이어야 함
_t0 = torch.zeros(4, dtype=torch.long, device=device)
assert (q_sample(_x0, _t0, torch.zeros_like(_x0)) - _x0).abs().max() < 0.05
print("문제 2 통과")

In [ ]:
# --- MNIST 준비 (다음 문제들에서도 사용) ---
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

tf = transforms.Compose([
    transforms.Pad(2),                       # 28x28 -> 32x32
    transforms.ToTensor(),                   # [0,1]
    transforms.Lambda(lambda x: x * 2 - 1),  # [-1,1] 로 정규화
])
train_full = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
train_set  = Subset(train_full, range(min(N_TRAIN, len(train_full))))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, drop_last=True)
print("학습 샘플 수:", len(train_set))

# --- forward diffusion 시각화 ---
x0 = train_set[0][0].unsqueeze(0).to(device)
ts = [0, 20, 50, 100, 150, 200, 300, T - 1]
noised = torch.cat([q_sample(x0, torch.tensor([t], device=device)) for t in ts], dim=0)
show_images(noised, title="Forward diffusion:  t = " + ", ".join(map(str, ts)), nrow=len(ts))

---
## 문제 3. 사후 분포 $q(\mathbf x_{t-1}\mid\mathbf x_t,\mathbf x_0)$ (난이도 ★★☆)

역과정 $p_\theta(\mathbf x_{t-1}\mid\mathbf x_t)$ 는 직접 알 수 없지만, **$\mathbf x_0$ 를 조건으로 주면**
사후 분포가 가우시안으로 닫힌 형식이 됩니다 (강의자료의 ELBO 유도에서 KL 항의 상대가 되는 분포).

$$q(\mathbf x_{t-1}\mid\mathbf x_t,\mathbf x_0)=\mathcal N\!\left(\mathbf x_{t-1};\tilde\mu_t(\mathbf x_t,\mathbf x_0),\ \tilde\beta_t\mathbf I\right)$$

$$\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t,\qquad
\tilde\mu_t=\underbrace{\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}}_{c_1}\mathbf x_0
+\underbrace{\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}}_{c_2}\mathbf x_t$$

또한 $\epsilon$ 예측 모델을 쓰기 위해 $\mathbf x_t$ 와 $\epsilon$ 로부터 $\mathbf x_0$ 를 되돌리는 식도 필요합니다.

$$\hat{\mathbf x}_0=\frac{\mathbf x_t-\sqrt{1-\bar\alpha_t}\,\epsilon}{\sqrt{\bar\alpha_t}}
=\sqrt{\tfrac{1}{\bar\alpha_t}}\,\mathbf x_t-\sqrt{\tfrac{1}{\bar\alpha_t}-1}\,\epsilon$$

### 할 일
1. `posterior_variance`, `posterior_mean_coef1`, `posterior_mean_coef2` 를 계산하세요.
2. `predict_x0_from_noise(x_t, t, noise)` 와 `q_posterior(x0, x_t, t)` 를 구현하세요.
3. 채점 셀은 아래 두 항등식을 확인합니다. 왜 성립하는지 설명하세요.
   - $c_1+c_2\sqrt{\bar\alpha_t}=\sqrt{\bar\alpha_{t-1}}$
   - $c_2^2(1-\bar\alpha_t)+\tilde\beta_t=1-\bar\alpha_{t-1}$


In [ ]:
# 문제 1과 같은 이유로 float64(...64 변수) 로 계산한 뒤 to32() 로 캐스팅한다.
# TODO: posterior_variance     = to32( betas64 * (1 - abar_prev64) / (1 - abar64) )
# TODO: posterior_mean_coef1   = to32( betas64 * sqrt(abar_prev64) / (1 - abar64) )
# TODO: posterior_mean_coef2   = to32( (1 - abar_prev64) * sqrt(alphas64) / (1 - abar64) )
raise NotImplementedError("여기를 구현하세요")

# t=0 에서 분산이 0 이 되므로 로그 계산 시 clamp 가 필요 (강의 코드의 posterior_log_variance_clipped)
posterior_log_variance_clipped = torch.log(posterior_variance.clamp(min=1e-20))


def predict_x0_from_noise(x_t, t, noise):
    """예측된 노이즈로부터 원본 x0 를 복원. return: (B,C,H,W)"""
    # TODO: extract(sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t
    # TODO:   - extract(sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
    raise NotImplementedError("여기를 구현하세요")


def q_posterior(x0, x_t, t):
    """q(x_{t-1} | x_t, x_0) 의 평균과 분산. return: (mean, var, log_var_clipped)"""
    # TODO: mean = coef1 * x0 + coef2 * x_t   (extract 사용)
    # TODO: var, log_var 도 extract 로 뽑아서 반환
    raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ===================== 문제 3 자동 채점 =====================
_c1, _c2 = posterior_mean_coef1, posterior_mean_coef2
_pv = posterior_variance

# (a) 평균에 대한 일관성:  c1 + c2*sqrt(abar_t) = sqrt(abar_{t-1})
_lhs = _c1 + _c2 * torch.sqrt(alphas_cumprod)
assert torch.allclose(_lhs, torch.sqrt(alphas_cumprod_prev), atol=1e-5), "c1, c2 식을 확인하세요"

# (b) 분산에 대한 일관성:  c2^2 (1-abar_t) + beta~_t = 1 - abar_{t-1}
_lhs = _c2 ** 2 * (1 - alphas_cumprod) + _pv
assert torch.allclose(_lhs, 1 - alphas_cumprod_prev, atol=1e-5), "posterior_variance 식을 확인하세요"
assert _pv[0].item() < 1e-8, "t=0 에서 사후 분산은 0 이어야 합니다"
assert torch.all(_pv <= betas + 1e-6), "beta~_t <= beta_t 여야 합니다"

# (c) x0 복원: q_sample 로 만든 x_t 와 사용한 noise 를 넣으면 원래 x0 가 나와야 함
_x0 = torch.randn(8, 1, 8, 8, device=device)
_t  = torch.randint(0, T, (8,), device=device)
_n  = torch.randn_like(_x0)
_xt = q_sample(_x0, _t, _n)
assert torch.allclose(predict_x0_from_noise(_xt, _t, _n), _x0, atol=1e-3), "x0 복원식을 확인하세요"

# (d) 몬테카를로로 사후 분포 검증 (1차원, 중요도 없이 조건부 직접 시뮬레이션)
_tv = 60
_NMC = 400000
_x0s = torch.zeros(_NMC, 1, 1, 1, device=device) + 0.3
_tt  = torch.full((_NMC,), _tv, device=device, dtype=torch.long)
_xtm1 = q_sample(_x0s, torch.full_like(_tt, _tv - 1))            # x_{t-1} ~ q(.|x0)
_xt   = torch.sqrt(alphas[_tv]) * _xtm1 + torch.sqrt(betas[_tv]) * torch.randn_like(_xtm1)
# x_t 가 특정 값 근방인 표본만 골라 조건부 분포를 근사
_target = 0.0
_mask = (_xt - _target).abs() < 0.01
_sel = _xtm1[_mask]
assert _sel.numel() > 500, "표본이 부족합니다 (다시 실행해 보세요)"
_mean_emp = _sel.mean().item()
_std_emp  = _sel.std().item()
_m_th, _v_th, _ = q_posterior(_x0s[:1], torch.full((1, 1, 1, 1), _target, device=device),
                              torch.tensor([_tv], device=device))
assert abs(_mean_emp - _m_th.item()) < 0.02, "사후 평균이 시뮬레이션과 다릅니다"
assert abs(_std_emp - _v_th.sqrt().item()) < 0.02, "사후 분산이 시뮬레이션과 다릅니다"
print("문제 3 통과  (표본 %d 개로 검증)" % _sel.numel())

**해설 (문제 3-3).**
두 항등식은 모두 **전체 기댓값/전체 분산의 법칙**을 사후 분포에 적용한 것입니다.
$\mathbf x_t\mid\mathbf x_0\sim\mathcal N(\sqrt{\bar\alpha_t}\mathbf x_0,(1-\bar\alpha_t)\mathbf I)$ 이므로

$$\mathbb E[\mathbf x_{t-1}\mid\mathbf x_0]=\mathbb E_{\mathbf x_t}\big[\tilde\mu_t\big]
=c_1\mathbf x_0+c_2\sqrt{\bar\alpha_t}\,\mathbf x_0 \stackrel{!}{=}\sqrt{\bar\alpha_{t-1}}\mathbf x_0
\;\Rightarrow\; c_1+c_2\sqrt{\bar\alpha_t}=\sqrt{\bar\alpha_{t-1}}$$

$$\mathrm{Var}[\mathbf x_{t-1}\mid\mathbf x_0]=\underbrace{c_2^2(1-\bar\alpha_t)}_{\text{평균의 분산}}+\underbrace{\tilde\beta_t}_{\text{조건부 분산의 평균}}\stackrel{!}{=}1-\bar\alpha_{t-1}$$

즉 사후 분포의 계수들은 forward process 와 **모순 없이** 맞물려 있어야 하고, 이 두 식이 그 정합성 검사입니다.


---
## 문제 4. 시간 임베딩 + U-Net + 학습 손실 $\lVert\epsilon-\epsilon_\theta(\mathbf x_t,t)\rVert^2$ (난이도 ★★★)

강의자료의 최종 학습 목표식은

$$L_{\text{simple}}=\mathbb E_{t\sim U[1,T],\ \mathbf x_0,\ \epsilon}\Big[\lambda(t)\big\lVert\epsilon-\epsilon_\theta(\mathbf x_t,t)\big\rVert^2\Big],
\qquad \mathbf x_t=\sqrt{\bar\alpha_t}\mathbf x_0+\sqrt{1-\bar\alpha_t}\epsilon$$

이고 실제 구현에서는 $\lambda(t)=1$ 로 둡니다. 즉 **"더해진 노이즈를 맞히는 회귀 문제"** 입니다.

모델 $\epsilon_\theta$ 는 U-Net 이며, 시점 $t$ 는 Transformer 의 위치 인코딩과 같은
**정현파(sinusoidal) 임베딩** 으로 각 residual block 에 주입됩니다.

$$\text{emb}(t)_{2i}=\sin\!\left(\frac{t}{10000^{2i/d}}\right),\qquad
\text{emb}(t)_{2i+1}=\cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

### 할 일
1. `SinusoidalPosEmb` 의 `forward` 를 구현하세요.
2. `p_losses(model, x0, t)` 를 구현하세요. (`q_sample` 재사용)
3. 학습을 돌리고 손실이 떨어지는지 확인하세요.


In [ ]:
class SinusoidalPosEmb(nn.Module):
    """시점 t (B,) -> (B, dim) 정현파 임베딩"""
    def __init__(self, dim):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t):
        # TODO: half = dim // 2
        # TODO: freqs = exp(-log(10000) * arange(half) / (half - 1))
        # TODO: args = t[:, None].float() * freqs[None, :]
        # TODO: return cat([sin(args), cos(args)], dim=-1)
        raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ---------------- 소형 U-Net (제공 코드, 수정 불필요) ----------------
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(F.silu(temb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SmallUNet(nn.Module):
    """32x32 입력용 소형 U-Net.  eps_theta(x_t, t) 를 출력한다."""
    def __init__(self, ch=64, in_ch=CHANNELS):
        super().__init__()
        time_dim = ch * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(ch), nn.Linear(ch, time_dim), nn.GELU(),
            nn.Linear(time_dim, time_dim),
        )
        self.init_conv = nn.Conv2d(in_ch, ch, 3, padding=1)
        # down: 32 -> 16 -> 8
        self.d1    = ResBlock(ch, ch, time_dim)
        self.down1 = nn.Conv2d(ch, ch * 2, 4, stride=2, padding=1)
        self.d2    = ResBlock(ch * 2, ch * 2, time_dim)
        self.down2 = nn.Conv2d(ch * 2, ch * 4, 4, stride=2, padding=1)
        # middle
        self.m1 = ResBlock(ch * 4, ch * 4, time_dim)
        self.m2 = ResBlock(ch * 4, ch * 4, time_dim)
        # up: 8 -> 16 -> 32  (skip connection 을 concat)
        self.up1 = nn.ConvTranspose2d(ch * 4, ch * 2, 4, stride=2, padding=1)
        self.u1  = ResBlock(ch * 4, ch * 2, time_dim)
        self.up2 = nn.ConvTranspose2d(ch * 2, ch, 4, stride=2, padding=1)
        self.u2  = ResBlock(ch * 2, ch, time_dim)
        self.out_norm = nn.GroupNorm(8, ch)
        self.out_conv = nn.Conv2d(ch, in_ch, 1)

    def forward(self, x, t):
        temb = self.time_mlp(t)
        x = self.init_conv(x)
        h1 = self.d1(x, temb)            # (B, ch, 32, 32)
        h  = self.down1(h1)
        h2 = self.d2(h, temb)            # (B, 2ch, 16, 16)
        h  = self.down2(h2)
        h  = self.m1(h, temb); h = self.m2(h, temb)
        h  = self.up1(h)                                    # (B, 2ch, 16, 16)
        h  = self.u1(torch.cat([h, h2], dim=1), temb)
        h  = self.up2(h)                                    # (B, ch, 32, 32)
        h  = self.u2(torch.cat([h, h1], dim=1), temb)
        return self.out_conv(F.silu(self.out_norm(h)))


model = SmallUNet(ch=BASE_CH).to(device)
print("파라미터 수: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))
with torch.no_grad():
    _o = model(torch.randn(2, CHANNELS, IMG_SIZE, IMG_SIZE, device=device),
               torch.tensor([0, T - 1], device=device))
print("출력 shape:", tuple(_o.shape))

In [ ]:
def p_losses(model, x0, t):
    """DDPM 학습 손실 L_simple.  return: 스칼라 텐서"""
    # TODO: noise = torch.randn_like(x0)
    # TODO: x_t   = q_sample(x0, t, noise)
    # TODO: pred  = model(x_t, t)
    # TODO: return F.mse_loss(pred, noise)
    raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ===================== 문제 4 자동 채점 =====================
# (a) 정현파 임베딩
_emb = SinusoidalPosEmb(64).to(device)
_t = torch.arange(10, device=device)
_e = _emb(_t)
assert _e.shape == (10, 64), "임베딩 출력 shape 은 (B, dim) 이어야 합니다"
assert torch.allclose(_e[0, :32], torch.zeros(32, device=device), atol=1e-6), "t=0 의 sin 성분은 0"
assert torch.allclose(_e[0, 32:], torch.ones(32, device=device), atol=1e-6), "t=0 의 cos 성분은 1"
assert not torch.allclose(_e[1], _e[2]), "서로 다른 t 는 다른 임베딩이어야 합니다"
assert _e.abs().max() <= 1.0 + 1e-6, "sin/cos 이므로 절댓값은 1 이하"

# (b) 손실 함수: 항상 0 을 출력하는 모델이면 손실 = E[eps^2] = 1
def _zero_model(x, t):
    return torch.zeros_like(x)
_l = p_losses(_zero_model, torch.randn(1024, 1, 8, 8, device=device),
              torch.randint(0, T, (1024,), device=device))
assert abs(_l.item() - 1.0) < 0.05, "eps 예측 MSE 라면 zero-model 손실은 1 근처여야 합니다"

# (c) x_t 를 만들 때 쓴 노이즈를 정확히 아는 오라클이면 손실 = 0
_x0 = torch.randn(64, 1, 8, 8, device=device).clamp(-1, 1)
_t  = torch.randint(1, T, (64,), device=device)
def _eps_oracle(x, t):
    return ((x - extract(sqrt_alphas_cumprod, t, x.shape) * _x0)
            / extract(sqrt_one_minus_alphas_cumprod, t, x.shape))
_l0 = p_losses(_eps_oracle, _x0, _t)
assert _l0.item() < 1e-4, "p_losses 내부에서 q_sample 로 만든 x_t 와 target noise 가 어긋나 있습니다"
print("문제 4 통과")

In [ ]:
# ---------------- 학습 ----------------
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
loss_history = []
t0 = time.time()

model.train()
for epoch in range(EPOCHS):
    running, nb = 0.0, 0
    for x, _ in train_loader:
        x = x.to(device)
        t = torch.randint(0, T, (x.shape[0],), device=device).long()
        loss = p_losses(model, x, t)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item(); nb += 1
    loss_history.append(running / nb)
    print("epoch %2d/%d | loss %.4f | %.1fs" % (epoch + 1, EPOCHS, loss_history[-1], time.time() - t0))

plt.figure(figsize=(5, 3))
plt.plot(loss_history, marker="o"); plt.xlabel("epoch"); plt.ylabel("L_simple")
plt.title("DDPM training loss"); plt.grid(alpha=.3); plt.show()

assert loss_history[-1] < loss_history[0], "손실이 감소하지 않았습니다 — 구현을 확인하세요"
assert loss_history[-1] < (0.5 if QUICK_MODE else 0.2), "손실이 충분히 낮지 않습니다 (에폭 수를 늘려보세요)"
print("학습 완료")

---
## 문제 5. 역과정 샘플링 $p_\theta(\mathbf x_{t-1}\mid\mathbf x_t)$ (난이도 ★★★)

학습이 끝나면 $\mathbf x_T\sim\mathcal N(0,\mathbf I)$ 에서 시작해 $t=T-1,\dots,0$ 으로 거슬러 올라가며 샘플링합니다.

$$\hat{\mathbf x}_0=\text{clip}\Big(\tfrac{\mathbf x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta(\mathbf x_t,t)}{\sqrt{\bar\alpha_t}},\,-1,\,1\Big),
\qquad
\mathbf x_{t-1}=\tilde\mu_t(\hat{\mathbf x}_0,\mathbf x_t)+\mathbb 1[t>0]\sqrt{\tilde\beta_t}\;\mathbf z,\quad \mathbf z\sim\mathcal N(0,\mathbf I)$$

$t=0$ 에서는 **노이즈를 더하지 않는다**는 점(강의 코드의 `no noise if t == 0`)에 주의하세요.

### 할 일
1. `p_sample(model, x, t_index)` 한 스텝을 구현하세요.
2. `p_sample_loop` 로 이미지를 생성하고, 중간 과정을 시각화하세요.


In [ ]:
@torch.no_grad()
def p_sample(model, x, t_index):
    """역과정 한 스텝: x_t -> x_{t-1}"""
    t = torch.full((x.shape[0],), t_index, device=x.device, dtype=torch.long)
    # TODO: eps  = model(x, t)
    # TODO: x0   = predict_x0_from_noise(x, t, eps) 후 [-1,1] 로 clamp
    # TODO: mean, var, _ = q_posterior(x0, x, t)
    # TODO: t_index == 0 이면 mean 을 그대로 반환, 아니면 mean + sqrt(var) * randn
    raise NotImplementedError("여기를 구현하세요")


@torch.no_grad()
def p_sample_loop(model, n=16, record_every=None):
    """x_T ~ N(0,I) 에서 시작해 x_0 까지 역과정 전체를 수행"""
    model.eval()
    x = torch.randn(n, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
    traj = []
    for i in reversed(range(T)):
        x = p_sample(model, x, i)
        if record_every is not None and (i % record_every == 0 or i == 0):
            traj.append(x[:1].clone())
    return (x, traj) if record_every is not None else x

In [ ]:
# ===================== 문제 5 자동 채점 =====================
# (a) 완벽한 모델(정답 노이즈를 아는 오라클)을 넣으면 역과정이 원본을 복원해야 한다
_x0_true = train_set[0][0].unsqueeze(0).to(device).repeat(4, 1, 1, 1)
torch.manual_seed(7)

class _Oracle(nn.Module):
    """x_t 를 만들 때 쓴 노이즈를 정확히 알고 있는 가상의 모델"""
    def __init__(self, x0): super().__init__(); self.x0 = x0
    def forward(self, x, t):
        ab  = extract(sqrt_alphas_cumprod, t, x.shape)
        sig = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
        return (x - ab * self.x0) / sig      # 정의상 eps = (x_t - sqrt(abar)x0)/sqrt(1-abar)

_oracle = _Oracle(_x0_true)
_x = torch.randn_like(_x0_true)
for _i in reversed(range(T)):
    _x = p_sample(_oracle, _x, _i)
_err = (_x - _x0_true).abs().mean().item()
assert _err < 0.05, f"오라클 모델로도 복원이 안 됩니다 (평균오차 {_err:.3f}) — p_sample 을 확인하세요"

# (b) t=0 스텝은 결정론적이어야 한다 (노이즈 추가 없음)
_xa = p_sample(_oracle, _x0_true.clone(), 0)
_xb = p_sample(_oracle, _x0_true.clone(), 0)
assert torch.allclose(_xa, _xb), "t=0 에서는 노이즈를 더하면 안 됩니다"

# (c) t>0 스텝은 확률적이어야 한다
_xa = p_sample(_oracle, _x0_true.clone(), 100)
_xb = p_sample(_oracle, _x0_true.clone(), 100)
assert not torch.allclose(_xa, _xb), "t>0 에서는 노이즈를 더해야 합니다"
print("문제 5 통과  (오라클 복원 오차 %.4f)" % _err)

In [ ]:
# ---------------- 실제 샘플 생성 ----------------
samples, traj = p_sample_loop(model, n=32, record_every=T // 8)
show_images(samples, title="DDPM samples on MNIST (T=%d steps)" % T, nrow=8)
show_images(torch.cat(traj, dim=0), title="Reverse process:  noise  ->  image",
            nrow=len(traj))

---
## 문제 6. 스코어 함수와 Langevin 동역학 (난이도 ★★☆)

강의자료의 두 번째 프레임워크인 **SGM(Score-based Generative Model)** 의 핵심은 Stein 스코어

$$\mathbf s_\theta(\mathbf x,t)\ \approx\ \nabla_{\mathbf x}\log q_t(\mathbf x)$$

입니다. Denoising score matching 유도식

$$\nabla_{\mathbf x_t}\log q(\mathbf x_t\mid\mathbf x_0)=-\frac{\mathbf x_t-\mathbf x_0\sqrt{\bar\alpha_t}}{1-\bar\alpha_t}=-\frac{\epsilon}{\sqrt{1-\bar\alpha_t}}$$

로부터, **DDPM 의 노이즈 예측 모델과 스코어 모델은 상수배 관계**임을 알 수 있습니다.

$$\boxed{\ \mathbf s_\theta(\mathbf x_t,t)=-\frac{\epsilon_\theta(\mathbf x_t,t)}{\sqrt{1-\bar\alpha_t}}\ }$$

그리고 스코어만 알면 강의자료의 **Langevin 동역학**으로 샘플을 뽑을 수 있습니다.

$$\mathbf x^{i+1}=\mathbf x^{i}+\frac{s}{2}\,\mathbf s_\theta(\mathbf x^{i})+\sqrt{s}\,\epsilon^{i},\qquad \epsilon^i\sim\mathcal N(0,\mathbf I)$$

### 할 일
1. `score_from_noise(eps, t)` 로 두 표현의 변환을 구현하세요.
2. 2차원 가우시안 혼합 분포의 **해석적 스코어** `score_gmm(x)` 를 구현하세요.
3. `langevin_sample` 로 그 분포에서 샘플을 뽑고, 실제 분포와 비교하세요.


In [ ]:
def score_from_noise(eps, t, x_shape=None):
    """eps 예측 -> 스코어 s = -eps / sqrt(1 - abar_t)"""
    x_shape = eps.shape if x_shape is None else x_shape
    # TODO: -eps / extract(sqrt_one_minus_alphas_cumprod, t, x_shape)
    raise NotImplementedError("여기를 구현하세요")


# --- 2D 가우시안 혼합 분포: p(x) = 0.5 N(mu1, s^2 I) + 0.5 N(mu2, s^2 I) ---
MU = torch.tensor([[-2.0, 0.0], [2.0, 0.0]])
SIG = 0.5

def score_gmm(x, mu=MU, sig=SIG):
    """해석적 스코어 grad_x log p(x).  x: (N,2) -> (N,2)

    log p(x) = logsumexp_k [ log(1/K) - ||x-mu_k||^2 / (2 sig^2) ] + const
    grad     = sum_k w_k(x) * (mu_k - x) / sig^2 ,  w_k = responsibility
    """
    mu = mu.to(x.device)
    # TODO: diff = x[:, None, :] - mu[None, :, :]              (N, K, 2)
    # TODO: logw = -(diff**2).sum(-1) / (2*sig**2)             (N, K)
    # TODO: w    = softmax(logw, dim=1)                        (N, K)
    # TODO: return (w[..., None] * (-diff) / sig**2).sum(1)
    raise NotImplementedError("여기를 구현하세요")


def langevin_sample(score_fn, n=4000, n_steps=1000, step=0.02, init_scale=3.0, seed=0):
    """Langevin 동역학:  x <- x + (step/2) * score(x) + sqrt(step) * eps"""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 2, generator=g) * init_scale
    # TODO: n_steps 번 반복하며 위 갱신식을 적용
    raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ===================== 문제 6 자동 채점 =====================
# (a) 스코어 <-> 노이즈 변환: q(x_t|x_0) 의 해석적 스코어와 일치하는가
_x0 = torch.randn(256, 1, 8, 8, device=device)
_t  = torch.randint(1, T, (256,), device=device)
_eps = torch.randn_like(_x0)
_xt = q_sample(_x0, _t, _eps)
_analytic = -(_xt - extract(sqrt_alphas_cumprod, _t, _xt.shape) * _x0) / \
            extract(1.0 - alphas_cumprod, _t, _xt.shape)
assert torch.allclose(score_from_noise(_eps, _t), _analytic, rtol=1e-3, atol=1e-3), \
    "score = -eps / sqrt(1-abar_t) 관계를 확인하세요"

# (b) GMM 스코어: 유한차분 미분과 비교
_x = torch.randn(50, 2) * 1.5
def _logp(x):
    d = x[:, None, :] - MU[None, :, :]
    return torch.logsumexp(-(d ** 2).sum(-1) / (2 * SIG ** 2), dim=1)
_h = 1e-3
_num = torch.zeros_like(_x)
for _j in range(2):
    _e = torch.zeros_like(_x); _e[:, _j] = _h
    _num[:, _j] = (_logp(_x + _e) - _logp(_x - _e)) / (2 * _h)
assert torch.allclose(score_gmm(_x), _num, atol=1e-2), "score_gmm 이 수치 미분과 다릅니다"
assert torch.allclose(score_gmm(MU.mean(0, keepdim=True)),
                      torch.zeros(1, 2), atol=1e-6), "대칭점에서 스코어는 0"

# (c) Langevin 샘플이 실제 분포를 재현하는가
_s = langevin_sample(score_gmm)
assert abs(_s.mean(0)[0].item()) < 0.3 and abs(_s.mean(0)[1].item()) < 0.2, "표본 평균이 (0,0) 근처여야 합니다"
_left  = (_s[:, 0] < 0).float().mean().item()
assert 0.3 < _left < 0.7, f"두 모드의 비율이 균형적이지 않습니다 (왼쪽 {_left:.2f})"
_sl = _s[_s[:, 0] < 0]; _sr = _s[_s[:, 0] >= 0]
assert abs(_sl[:, 0].mean().item() + 2.0) < 0.25, "왼쪽 모드 중심이 -2 근처여야 합니다"
assert abs(_sr[:, 0].mean().item() - 2.0) < 0.25, "오른쪽 모드 중심이 +2 근처여야 합니다"
assert abs(_s[:, 1].std().item() - SIG) < 0.15, "모드 내 표준편차가 sigma 와 달라요"
print("문제 6 통과  (왼쪽 모드 비율 %.2f)" % _left)

In [ ]:
# 시각화 — 스코어 벡터장과 Langevin 샘플
gx, gy = torch.meshgrid(torch.linspace(-4, 4, 22), torch.linspace(-2.5, 2.5, 16), indexing="ij")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
sc = score_gmm(grid)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].quiver(grid[:, 0], grid[:, 1], sc[:, 0], sc[:, 1], angles="xy", scale=120, width=.003)
ax[0].set_title(r"score field  $\nabla_x \log p(x)$"); ax[0].set_aspect("equal")

s = langevin_sample(score_gmm)
ax[1].scatter(s[:, 0], s[:, 1], s=3, alpha=.25)
ax[1].set_xlim(-4, 4); ax[1].set_ylim(-2.5, 2.5); ax[1].set_aspect("equal")
ax[1].set_title("Langevin dynamics samples")
plt.tight_layout(); plt.show()

**참고 — SDE 관점 (강의자료 2.3절).**
Langevin 갱신식은 사실 역시간 SDE

$$\mathrm d\mathbf x=\big[f(\mathbf x,t)-g^2(t)\nabla_{\mathbf x}\log q_t(\mathbf x)\big]\mathrm dt+g(t)\mathrm d\bar{\mathbf w}$$

의 이산화(Euler–Maruyama)에 해당합니다.
DDPM 은 $\mathrm d\mathbf x=-\tfrac12\beta(t)\mathbf x\,\mathrm dt+\sqrt{\beta(t)}\,\mathrm d\mathbf w$ (VP-SDE),
SMLD/SGM 은 $\mathrm d\mathbf x=\sqrt{\tfrac{\mathrm d[\sigma^2(t)]}{\mathrm dt}}\,\mathrm d\mathbf w$ (VE-SDE) 의 이산화이며,
문제 5(DDPM 샘플링)와 문제 6(Langevin)이 **같은 이론의 두 이산화**라는 점이 이 장의 핵심 메시지입니다.


---
## 문제 7 (보너스). DDIM 가속 샘플링 (난이도 ★★★)

DDPM 샘플링은 $T$ 번의 신경망 호출(NFE)이 필요해 느립니다.
DDIM 은 마르코프 가정을 버리고 **부분수열** $\{\tau_1<\dots<\tau_S\}\subset\{1,\dots,T\}$ 만 사용합니다.

$$\mathbf x_{\tau_{i-1}}=\sqrt{\bar\alpha_{\tau_{i-1}}}\,\hat{\mathbf x}_0
+\sqrt{1-\bar\alpha_{\tau_{i-1}}-\sigma_i^2}\;\epsilon_\theta(\mathbf x_{\tau_i},\tau_i)+\sigma_i\mathbf z$$

$$\sigma_i=\eta\sqrt{\frac{1-\bar\alpha_{\tau_{i-1}}}{1-\bar\alpha_{\tau_i}}}\sqrt{1-\frac{\bar\alpha_{\tau_i}}{\bar\alpha_{\tau_{i-1}}}}$$

$\eta=0$ 이면 **결정론적**(같은 $\mathbf x_T$ → 같은 결과), $\eta=1$ 이면 DDPM 과 동일합니다.

### 할 일
1. `ddim_sample` 을 구현하세요.
2. NFE = 20 / 50 / 400 을 비교해 품질과 속도를 관찰하세요.
3. $\eta=0$ 일 때 결정론적임을 확인하세요.


In [ ]:
@torch.no_grad()
def ddim_sample(model, n=16, n_steps=50, eta=0.0, x_T=None):
    model.eval()
    x = torch.randn(n, CHANNELS, IMG_SIZE, IMG_SIZE, device=device) if x_T is None else x_T.clone()
    times = torch.linspace(T - 1, 0, n_steps).round().long().tolist()
    # TODO: for i, t_cur in enumerate(times):
    # TODO:     t_prev = times[i+1] if i+1 < len(times) else -1
    # TODO:     eps = model(x, t 벡터)
    # TODO:     ab = alphas_cumprod[t_cur],  ab_prev = alphas_cumprod[t_prev] (t_prev<0 이면 1.0)
    # TODO:     x0 = ((x - sqrt(1-ab)*eps) / sqrt(ab)).clamp(-1,1)
    # TODO:     sigma = eta * sqrt((1-ab_prev)/(1-ab)) * sqrt(1 - ab/ab_prev)
    # TODO:     c = sqrt(clamp(1 - ab_prev - sigma^2, min=0))
    # TODO:     x = sqrt(ab_prev)*x0 + c*eps + sigma*randn  (마지막 스텝이면 노이즈 없이)
    raise NotImplementedError("여기를 구현하세요")

In [ ]:
# ===================== 문제 7 자동 채점 =====================
_xT = torch.randn(8, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
_a = ddim_sample(model, n=8, n_steps=20, eta=0.0, x_T=_xT)
_b = ddim_sample(model, n=8, n_steps=20, eta=0.0, x_T=_xT)
assert _a.shape == (8, CHANNELS, IMG_SIZE, IMG_SIZE)
assert torch.allclose(_a, _b, atol=1e-5), "eta=0 이면 결정론적이어야 합니다"
_c = ddim_sample(model, n=8, n_steps=20, eta=1.0, x_T=_xT)
assert not torch.allclose(_a, _c, atol=1e-3), "eta=1 이면 확률적이어야 합니다"
assert _a.abs().max().item() < 3.0, "출력 범위가 비정상적으로 큽니다"
# 스텝 수를 늘리면 결과가 달라져야 함 (같은 x_T 라도 궤적이 다름)
_d = ddim_sample(model, n=8, n_steps=100, eta=0.0, x_T=_xT)
assert not torch.allclose(_a, _d, atol=1e-3)
print("문제 7 통과")

In [ ]:
# NFE 비교
for nfe in [20, 50, 200]:
    t0 = time.time()
    imgs = ddim_sample(model, n=16, n_steps=nfe, eta=0.0)
    show_images(imgs, title="DDIM  NFE=%d  (%.1fs)" % (nfe, time.time() - t0), nrow=8)

t0 = time.time()
imgs = p_sample_loop(model, n=16)
show_images(imgs, title="DDPM  NFE=%d  (%.1fs)" % (T, time.time() - t0), nrow=8)

---
## 마무리 — 정리 문제 (서술)

1. DDPM 의 ELBO 유도에서 $\mathbb E_q[D_{KL}(q(\mathbf x_{t-1}\mid\mathbf x_t,\mathbf x_0)\Vert p_\theta(\mathbf x_{t-1}\mid\mathbf x_t))]$ 항이
   왜 결국 $\lVert\epsilon-\epsilon_\theta\rVert^2$ 형태의 단순한 MSE 로 정리되는지 설명하시오.
   *(힌트: 두 분포 모두 분산이 고정된 가우시안이면 KL 은 평균 차이의 제곱이 되고, $\tilde\mu_t$ 를 $\epsilon$ 로 다시 쓰면 상수배만 남는다.)*
2. DDPM(문제 5)과 Langevin 동역학(문제 6)이 SDE 관점에서 어떻게 통합되는지 서술하시오.
3. 강의자료 그림 2-7 의 U-Net 에서 skip connection 과 timestep embedding 이 각각 어떤 역할을 하는지,
   문제 4의 `SmallUNet` 코드와 대응시켜 설명하시오.
4. DDIM 에서 $\eta$ 를 0 으로 두면 생성이 결정론적이 됩니다. 이 성질이 이미지 편집·보간(interpolation)에
   왜 유용한지 설명하시오.
